# YuE2 — standard ComfyUI canvas (revision-15 runtime)

This is the plain ComfyUI canvas notebook, now carrying the **same repaired
runtime as YuE2_Music_Revision15.ipynb**: the pinned ComfyUI v0.43.2 checkout,
the wheelhouse-first installer that fetches only missing wheels from PyPI, the
repaired native FP16 engine sources, and the **resumable, checksum-verified
cover upload**. Attach your existing **yue2-music-assets** dataset, choose
**GPU T4 x2**, enable **Internet**, and run the one code cell below. It prints
two Cloudflare links: the **Full-song canvas** and the **Cover canvas**. Nothing
runs until you press **Queue Prompt** inside ComfyUI. Keep the cell running.

**Cover upload for big WAV files:** ComfyUI's own Load Audio control sends one
request and the server rejects anything over **100 MB**. Instead, press
**Upload song (resumable)** on the LoadAudio node in the cover canvas. Your file
travels in verified 1 MB pieces (up to **2 GiB**): every piece carries its
SHA-256, a corrupted piece is refused instead of saved, finished pieces are kept
if the upload is interrupted, and choosing the same file again resumes. The
finished song's SHA-256 is printed in the browser console and recorded with the
upload, so nothing is lost or silently altered. Your WAV is kept unchanged.

The song canvas chain is native engine + style/lyrics + settings → plan → music
semantic tokens → acoustic synthesis → stereo decoder → save masters / audio
preview, as separate executing stage nodes. Native FP16/CUDA-graph generation,
16 Euler steps, CFG 1 and the 8-minute safety ceiling remain. Saved songs,
tokens and latents stay in `/kaggle/working/yue2_native/runs`; download them or
**Save Version with outputs** before ending the session.

Long-song quality, runtime and GPU peaks remain unverified; CPU/schema checks do
not prove GPU generation. The old pre-revision-15 canvas is not installed by
this notebook anymore.


In [ ]:
import base64, json, os, sys, zlib
from pathlib import Path
if "app" in globals():
    app.stop()  # Release a managed server from the earlier ComfyUI notebook.
if "native_process" in globals() and native_process.poll() is None:
    native_process.terminate()
    try:
        native_process.wait(timeout=20)
    except Exception:
        native_process.kill()
        native_process.wait(timeout=10)
support = Path("/kaggle/temp/yue2_comfy_support")
support.mkdir(parents=True, exist_ok=True)
payload = "..."
assert len(payload) > 1000, "Embedded canvas source payload is missing"
for relative, contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    relative = str(relative).replace(chr(92), "/")
    path = support / relative
    if not path.resolve().is_relative_to(support.resolve()):
        raise RuntimeError("Invalid embedded source path")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
for name in list(sys.modules):
    if name.split(".")[0] in {"native_yue2", "kaggle_yue2"}:
        del sys.modules[name]
if str(support) not in sys.path:
    sys.path.insert(0, str(support))
from native_yue2.setup import install
ROOT = "/kaggle/temp/yue2_native"
OUTPUTS = "/kaggle/working/yue2_native/runs"
ASSET_DATASET = "auto"  # Attach saved YuE2 assets with Add Input. No download fallback.
native_python, native_env = install(ROOT, OUTPUTS, assets=ASSET_DATASET)

# --- start the persistent ComfyUI server --------------------------------------
import json as _json
import re as _re
import socket as _socket
import subprocess as _subprocess
import threading as _threading
import time as _time
import urllib.request as _urlrequest
from IPython.display import HTML, display
from native_yue2.setup import COMFY_PORT, install_comfyui
from native_yue2.tunnel import CloudflareAccess

runtime_dir = Path(OUTPUTS).parent / "runtime"
runtime_dir.mkdir(parents=True, exist_ok=True)


def _summarize_runs():
    for directory in sorted(Path(OUTPUTS).glob("*")):
        if directory.is_dir():
            status = directory / "status.json"
            note = (status.read_text(errors="replace")[:180] if status.exists() else "Partial run")
            print("run:", directory.name, "—", note, flush=True)
    for archive in sorted(Path(OUTPUTS).glob("*.zip")):
        print("zip:", archive.name, flush=True)


def comfy_ready(port=COMFY_PORT):
    try:
        with _urlrequest.urlopen(f"http://127.0.0.1:{port}/system_stats", timeout=3) as response:
            return response.status == 200
    except Exception:
        return False


if "comfy_tunnel" in globals() and comfy_tunnel is not None:
    comfy_tunnel.close()
if "comfy_process" in globals() and comfy_process.poll() is None:
    comfy_process.terminate()
    try:
        comfy_process.wait(timeout=15)
    except _subprocess.TimeoutExpired:
        comfy_process.kill()
        comfy_process.wait(timeout=10)

if comfy_ready():
    raise RuntimeError("A ComfyUI server is already running on port 8188. Stop its owning cell before starting this notebook.")
else:
    with _socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", COMFY_PORT)) == 0:
            raise RuntimeError(f"Port {COMFY_PORT} is occupied by another program.")
    comfy_dir, comfy_env = install_comfyui(ROOT, OUTPUTS, native_env)
    log_path = runtime_dir / "comfyui.log"
    comfy_log = log_path.open("a")
    comfy_process = _subprocess.Popen(
        [native_python, "-u", str(Path(comfy_dir) / "main.py"), "--listen", "127.0.0.1", "--port", str(COMFY_PORT),
         "--output-directory", str(OUTPUTS), "--disable-auto-launch", "--preview-method", "none"],
        cwd=str(comfy_dir), env=comfy_env, stdout=_subprocess.PIPE,
        stderr=_subprocess.STDOUT, text=True, bufsize=1)

    def _drain(stream, log):
        for line in stream:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        log.close()

    _threading.Thread(target=_drain, args=(comfy_process.stdout, comfy_log), daemon=True).start()
    for _ in range(150):
        if comfy_process.poll() is not None:
            raise RuntimeError("ComfyUI exited during startup; inspect the log above.")
        if comfy_ready():
            break
        _time.sleep(2)
    else:
        raise TimeoutError("ComfyUI did not become ready; inspect the log above.")

object_info = _json.loads(_urlrequest.urlopen(
    f"http://127.0.0.1:{COMFY_PORT}/object_info", timeout=15).read().decode())
missing = [name for name in ("YuE2RunContext", "YuE2LoadEngine", "YuE2SaveSong") if name not in object_info]
if missing:
    raise RuntimeError(f"KaggleYuE2 nodes missing from the server: {missing}; inspect the startup log.")

comfy_tunnel = CloudflareAccess(
    ROOT, assets=native_env.get("YUE2_ASSET_DATASET"),
    log=runtime_dir / "comfy-cloudflared.log")
base_link = None
try:
    base_link = comfy_tunnel.start(COMFY_PORT, ready_path="/system_stats")
except Exception as exc:
    print(f"Cloudflare is unavailable: {exc}\nThe local server keeps running on port {COMFY_PORT}.", flush=True)
print("\nIn ComfyUI: Workflows → YuE2 Full Song / YuE2 Cover · or press Run on the opened canvas", flush=True)
if base_link:
    song_link = base_link + "/?yue2_workflow=song"
    cover_link = base_link + "/?yue2_workflow=cover"
    print("Full-song canvas: ", song_link, flush=True)
    print("Cover canvas:     ", cover_link, flush=True)
    display(HTML('<a href="' + song_link + '" target="_blank" rel="noopener noreferrer">Open the YuE2 full-song canvas</a><br>'
                 '<a href="' + cover_link + '" target="_blank" rel="noopener noreferrer">Open the YuE2 cover canvas</a><br>'
                 '<a href="' + base_link + '" target="_blank" rel="noopener noreferrer">Open plain ComfyUI</a>'))
print("Saved songs so far:", flush=True)
_summarize_runs()
print("Keep this Kaggle session and this cell running while using ComfyUI.", flush=True)
print("Generate from the canvas: Queue Prompt. Stop interrupts at a step boundary and saves earlier stages.", flush=True)

try:
    while comfy_process.poll() is None:
        _time.sleep(5)
    raise RuntimeError("ComfyUI server stopped; inspect the log above.")
finally:
    comfy_tunnel.close()
    if comfy_process.poll() is None:
        comfy_process.terminate()
        try:
            comfy_process.wait(timeout=15)
        except _subprocess.TimeoutExpired:
            comfy_process.kill()
            comfy_process.wait(timeout=5)
